# Modeling: Baseline


## Purpose
Train logistic regression and naive baselines. Establish the performance floor that all advanced models must beat.

### Historical Benchmarks
- Naive (always pick higher seed): ~71% accuracy
- Logistic Regression: typically ~73-75%
- Good model target: >76% accuracy, log_loss < 0.52

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys; sys.path.insert(0, str(Path(".").parent / "src"))
from nba_predictor.config import cfg
from nba_predictor.models.baseline import run_cv_baseline, get_feature_cols
from nba_predictor.evaluation.cv_strategy import get_cv_fold_summary


## Load Data and Inspect CV Structure

In [ ]:
series_path = cfg.project_root / "data/processed/series_dataset.parquet"
if not series_path.exists():
    raise RuntimeError("Run make process first")
df = pd.read_parquet(series_path)
print(f"Dataset: {len(df)} series, seasons {df.season.min()}–{df.season.max()}")
print("
CV fold structure:")
print(get_cv_fold_summary(df).to_string())


## Naive Baseline: Always Pick Higher Seed

In [ ]:
# Compute naive accuracy
naive_acc = df["higher_seed_wins"].mean()
print(f"Naive accuracy (always higher seed): {naive_acc:.3f}")
print(f"This is the floor — any model must beat {naive_acc:.1%}")


## Logistic Regression Walk-Forward CV

In [ ]:
cv_metrics = run_cv_baseline(df)
print(f"Accuracy: {np.mean(cv_metrics["accuracy"]):.3f} ± {np.std(cv_metrics["accuracy"]):.3f}")
print(f"Log-loss: {np.mean(cv_metrics["log_loss"]):.3f} ± {np.std(cv_metrics["log_loss"]):.3f}")
print(f"Brier:    {np.mean(cv_metrics["brier_score"]):.3f} ± {np.std(cv_metrics["brier_score"]):.3f}")
print(f"ECE:      {np.mean(cv_metrics["ece"]):.3f} ± {np.std(cv_metrics["ece"]):.3f}")


## Calibration Analysis

In [ ]:
# Plot reliability diagram: predicted probability vs actual win rate
# Well-calibrated model: diagonal line
